# 코랩에서 클라우드 터널로 파일을 내 PC로 받아보자

Python HTTP 서버 + cloudflared quick-tunnel로 Drive 파일을 PC wget 한 줄로 받습니다.
셀을 위에서 아래로 순서대로 실행하세요.

In [ ]:
# 1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 서비스 설정
PORT = 8080
SERVE_DIR = '/content/drive/MyDrive/downloads'  # 서빙할 Drive 폴더 경로 교체

In [ ]:
# 3. HTTP 서버 + cloudflared 터널 시작
import os, re, subprocess, threading
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

# 서빙 폴더가 없으면 생성
Path(SERVE_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(SERVE_DIR)

# HTTP 서버 백그라운드 실행
httpd = HTTPServer(('0.0.0.0', PORT), SimpleHTTPRequestHandler)
t = threading.Thread(target=httpd.serve_forever, daemon=True)
t.start()
print(f'HTTP 서버 시작: http://localhost:{PORT}')

# cloudflared 바이너리 내려받기
CF_BIN = '/content/cloudflared'
if not os.path.exists(CF_BIN):
    print('cloudflared 다운로드 중...')
    subprocess.run(
        ['wget', '-q', '-O', CF_BIN,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        check=True
    )
    os.chmod(CF_BIN, 0o755)
    print('cloudflared 준비 완료')

# 터널 실행 — stderr에서 URL 파싱
proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', f'http://localhost:{PORT}'],
    stderr=subprocess.PIPE,
    text=True
)
tunnel_url = None
for line in proc.stderr:
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group()
        break

if tunnel_url is None:
    print('\n오류: 터널 URL을 찾지 못했습니다. cloudflared 실행 오류를 확인하세요.')
else:
    print(f'\n터널 URL: {tunnel_url}')
    print('\nPC에서 아래 명령으로 파일을 받으세요:')
    print(f"  wget '{tunnel_url}/파일명'")
    print('\n※ 이 셀을 유지한 채로 사용하세요. 셀을 중단하면 터널이 닫힙니다.')

In [ ]:
# 4. 터널 종료 (파일 전송 완료 후 실행)
proc.terminate()
httpd.shutdown()
print('터널과 HTTP 서버를 종료했습니다.')